In [ ]:
!pip install torch torchvision torchaudio scikit-learn tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import numpy as np
import torch

ROOT        = "/content/drive/MyDrive/Hackenza_MUPlovers"
FEATURE_DIR = os.path.join(ROOT, "cache/features_normalized")  # ← normalized features
TRAIN_MANIFEST = os.path.join(ROOT, "data/train_manifest_split.csv")
VAL_MANIFEST   = os.path.join(ROOT, "data/val_manifest_split.csv")

print("Feature dir exists:", os.path.exists(FEATURE_DIR))
print("Train manifest exists:", os.path.exists(TRAIN_MANIFEST))
print("Val manifest exists:", os.path.exists(VAL_MANIFEST))

Feature dir exists: True
Train manifest exists: True
Val manifest exists: True


In [ ]:
import os

data_path = os.path.join(ROOT, "data")
print("Files inside data/:")
print(os.listdir(data_path))

Files inside data/:
['train_manifest.csv', 'chunk_index.csv', 'chunk_vad.csv', 'processed', 'test_ids.csv', 'train_ids.gsheet', 'test_manifest_split.csv', 'val_manifest_split.csv', 'train_manifest_split.csv', 'external']


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
from torch.utils.data import Dataset

class NativeDataset(Dataset):
    def __init__(self, manifest_csv, feature_dir):
        self.df = pd.read_csv(manifest_csv)
        self.feature_dir = feature_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_id = str(row["file_id"])
        label   = float(row["label"])

        feature_path = os.path.join(self.feature_dir, f"{file_id}.npy")
        x = np.load(feature_path)
        x = torch.tensor(x, dtype=torch.float32)

        return x, torch.tensor(label, dtype=torch.float32)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    sequences, labels = zip(*batch)

    lengths = [seq.shape[0] for seq in sequences]
    padded  = pad_sequence(sequences, batch_first=True)

    mask = torch.zeros(padded.shape[:2], dtype=torch.float32)
    for i, length in enumerate(lengths):
        mask[i, :length] = 1.0

    labels = torch.stack(labels)

    return padded, mask, labels

In [ ]:
from torch.utils.data import DataLoader

train_dataset = NativeDataset(TRAIN_MANIFEST, FEATURE_DIR)
val_dataset   = NativeDataset(VAL_MANIFEST,   FEATURE_DIR)

train_loader = DataLoader(
    train_dataset,
    batch_size  = 8,
    shuffle     = True,
    collate_fn  = collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size  = 8,
    shuffle     = False,
    collate_fn  = collate_fn
)

print("Train size:", len(train_dataset))
print("Val size  :", len(val_dataset))

Train size: 112
Val size  : 24


In [ ]:
# Automatically compute class weight from training data
train_df = pd.read_csv(TRAIN_MANIFEST)
num_native     = (train_df['label'] == 1).sum()
num_non_native = (train_df['label'] == 0).sum()

pos_weight = torch.tensor([num_non_native / num_native], dtype=torch.float32).to(device)

print(f"Native samples    : {num_native}")
print(f"Non-native samples: {num_non_native}")
print(f"pos_weight        : {pos_weight.item():.4f}")

Native samples    : 80
Non-native samples: 32
pos_weight        : 0.4000


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class AccentClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_heads=4, dropout=0.3):
        super().__init__()

        # 1️⃣ Projection layer
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )

        # 2️⃣ 2-layer BiGRU
        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        # 3️⃣ Multi-head attention
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim * 2,
            num_heads=num_heads,
            batch_first=True
        )

        # 4️⃣ BatchNorm
        self.batchnorm = nn.BatchNorm1d(hidden_dim * 2)

        # 5️⃣ Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, mask=None):
        # x: [batch, seq_len, input_dim]
        x = self.projection(x)              # [B, T, H]

        gru_out, _ = self.gru(x)            # [B, T, 2H]

        if mask is not None:
            # key_padding_mask: True for padded positions
            key_padding_mask = (mask == 0)  # [B, T]
            attn_out, _ = self.attention(
                gru_out, gru_out, gru_out,
                key_padding_mask=key_padding_mask
            )
            # masked mean pooling
            mask_exp = mask.unsqueeze(-1)   # [B, T, 1]
            attn_out = attn_out * mask_exp
            lengths = mask.sum(dim=1, keepdim=True).clamp(min=1e-6)
            pooled = attn_out.sum(dim=1) / lengths
        else:
            attn_out, _ = self.attention(gru_out, gru_out, gru_out)
            pooled = attn_out.mean(dim=1)   # [B, 2H]

        pooled = self.batchnorm(pooled)
        logits = self.classifier(pooled)    # [B, 1]
        return logits.squeeze(1)


In [ ]:
import os

feature_files = set(os.listdir(FEATURE_DIR))

import pandas as pd
train_df = pd.read_csv(TRAIN_MANIFEST)

missing = []
for idx in train_df["file_id"]:   # replace "id" with actual column name if different
    fname = f"{idx}.npy"
    if fname not in feature_files:
        missing.append(fname)

print("Missing files:", len(missing))
print(missing[:10])

Missing files: 0
[]


In [ ]:
import torch.optim as optim

# Dynamically infer input dimension (Phase 1)
sample_batch = next(iter(train_loader))
sample_x, sample_mask, sample_y = sample_batch
input_dim = sample_x.shape[-1]

print("Detected input_dim:", input_dim)  # should be 1024 + prosody_dim + noise_dim

model = AccentClassifier(input_dim=input_dim).to(device)  # ← new model
optimizer = optim.Adam(model.parameters(), lr=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience=3,
    factor=0.5,
)

print("✅ Model ready!")
print("Total parameters:", sum(p.numel() for p in model.parameters()))


Detected input_dim: 1039
✅ Model ready!
Total parameters: 3422209


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

def train_epoch():
    model.train()
    total_loss = 0

    for x, mask, y in tqdm(train_loader):
        x, mask, y = x.to(device), mask.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x, mask)

        # ← pos_weight added here
        loss = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(train_loader)


def evaluate():
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for x, mask, y in val_loader:
            x, mask, y = x.to(device), mask.to(device), y.to(device)

            logits = model(x, mask)
            loss   = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    acc     = accuracy_score(all_labels, all_preds)
    f1      = f1_score(all_labels, all_preds, zero_division=0)
    val_loss = total_loss / len(val_loader)

    return acc, f1, val_loss

In [ ]:
EPOCHS   = 50
best_acc = 0
best_f1  = 0

for epoch in range(EPOCHS):
    loss              = train_epoch()
    acc, f1, val_loss = evaluate()

    # Step scheduler based on val loss
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss : {loss:.4f}")
    print(f"  Val Loss   : {val_loss:.4f}")
    print(f"  Val Acc    : {acc:.4f}")
    print(f"  Val F1     : {f1:.4f}")

    # Save best model
    if acc > best_acc:
        best_acc = acc
        best_f1  = f1
        os.makedirs('/content/drive/MyDrive/Hackenza_MUPlovers/runs/', exist_ok=True)
        torch.save(model.state_dict(), '/content/drive/MyDrive/Hackenza_MUPlovers/runs/best_model.pt')
        print(f"  ✅ Best model saved!")

    print("-" * 40)

print(f"\n🏆 Best Accuracy: {best_acc:.4f}")
print(f"🏆 Best F1      : {best_f1:.4f}")

100%|██████████| 14/14 [00:04<00:00,  3.10it/s]


Epoch 1/50
  Train Loss : 0.3894
  Val Loss   : 0.3925
  Val Acc    : 0.7083
  Val F1     : 0.7407
  ✅ Best model saved!
----------------------------------------


100%|██████████| 14/14 [00:00<00:00, 21.44it/s]


Epoch 2/50
  Train Loss : 0.2922
  Val Loss   : 0.3674
  Val Acc    : 0.6667
  Val F1     : 0.7333
----------------------------------------


100%|██████████| 14/14 [00:00<00:00, 22.10it/s]


...

Epoch 48/50
  Train Loss : 0.0147
  Val Loss   : 0.5644
  Val Acc    : 0.6250
  Val F1     : 0.7097
----------------------------------------


100%|██████████| 14/14 [00:00<00:00, 21.98it/s]


Epoch 49/50
  Train Loss : 0.0303
  Val Loss   : 0.5808
  Val Acc    : 0.6250
  Val F1     : 0.7097
----------------------------------------


100%|██████████| 14/14 [00:00<00:00, 22.70it/s]


Epoch 50/50
  Train Loss : 0.0103
  Val Loss   : 0.5639
  Val Acc    : 0.6667
  Val F1     : 0.7500
----------------------------------------

🏆 Best Accuracy: 0.7917
🏆 Best F1      : 0.8485
